# Create HAL configs and shutters for the bits rounds -- variable z per FOV

Based on notebook 01 (same `get_frame_table`/`create_shutter_file`/`create_hal_config`
pipeline), but instead of one fixed z-range for every FOV, this reads notebook
04's per-FOV z table (`metadata/z_per_fov_table.csv`) and buckets FOVs into a
small number of discrete z-depth **tiers**. Each tier gets its own complete
hal_config + shutter (a full, independent frame table, generated the same way
notebook 01's single "bits" block used to) -- so a thin-tissue FOV can use a
noticeably shorter (and therefore faster/smaller) movie than a thick one,
instead of every FOV in the round imaging the worst-case depth.

**How this actually works on the microscope** (see
`../../../../../misc/dave_multi_z/README.md` for the full investigation):
Dave's positions file gains an optional 3rd column naming, per FOV, which
already-loaded HAL parameter set (hal_config) to use for that position --
this is HAL's own normal `Set Parameters` -> `Take Movie` cycle, just
switched per-FOV instead of manually per-round. You still load each tier's
hal_config into HAL's window before the round starts (same manual step as
today, just with `N_TIERS` files instead of one) -- **not** something this
notebook can do for you.

**Where files go**: tier hal_configs + shutters are written to a dedicated
`SAMPLE_DIR/multi_z/` folder, kept *together* (not split further) so HAL's
own same-directory `<shutters>` resolution keeps working exactly as it does
for the regular `settings/` layout -- see the folder-layout discussion in
that same README. Frame tables still go to `metadata/`, like every other
round's.

In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd

MERCI_DIR  = Path(os.getcwd()).parent.parent.parent.parent   # MERci/ (notebook lives in MERci/notebooks/prepare_imaging/<variant>/<acquisition>/)
SAMPLE_DIR = MERCI_DIR.parent                  # experiment root, e.g. LT048_sample_18/
sys.path.insert(0, str(MERCI_DIR / "src"))

from MERci.acquisition.configs import (
    get_frame_table, get_color_sequence_name,
    create_shutter_file, create_hal_config, power_dict_to_channel_list,
    frame_table_filename, shutter_filename, hal_config_filename,
)
from MERci.acquisition.display import print_frame_table, display_xml
from MERci.visualization       import visualize_shutter_sequence
from MERci.common.experiment_info import resolve_sample_identity, positions_file_tag

In [ ]:
SETTINGS_DIR = SAMPLE_DIR / "settings"     # regular (cells/transit) configs -- unaffected
METADATA_DIR = SAMPLE_DIR / "metadata"
POSITIONS_DIR = SAMPLE_DIR / "positions"
MULTI_Z_DIR  = SAMPLE_DIR / "multi_z"      # NEW: tier hal_configs + shutters live here instead
MULTI_Z_DIR.mkdir(parents=True, exist_ok=True)

# SAMPLE_NAME/IMAGING_DIR resolved from folder structure, NOT SAMPLE_DIR.name --
# POSITIONS_TAG is what positions_*.txt filenames below actually use.
SAMPLE_NAME, IMAGING_DIR = resolve_sample_identity(MERCI_DIR)
POSITIONS_TAG = positions_file_tag(SAMPLE_NAME, IMAGING_DIR)

MICROSCOPE = "ST2"   # must match notebook 01

_hal_dir        = MERCI_DIR / "data" / "configs" / "hal"
_hal_candidates = sorted(
    p for p in _hal_dir.glob("hal-config-*.xml")
    if MICROSCOPE.lower() in p.name.lower()
)
if not _hal_candidates:
    raise FileNotFoundError(f"No HAL template found for microscope '{MICROSCOPE}' in {_hal_dir}")
HAL_TEMPLATE = _hal_candidates[0]

FILE_TYPE     = ".zarr"
EXPOSURE_TIME = 0.15      # seconds -- must match notebook 01's bits round exposure

# Same POWER convention as notebook 01 -- only list wavelengths this
# microscope actually has (ST2/MFX: 650/560/488/405, no 750).
POWER = {
    650: 1.00,
    560: 1.00,
    488: 1.00,
    405: 1.00,
}
POWER_DEFAULT = 1.0

print(f"SAMPLE_DIR   : {SAMPLE_DIR}")
print(f"SAMPLE_NAME  : {SAMPLE_NAME}")
print(f"POSITIONS_TAG: {POSITIONS_TAG}")
print(f"MULTI_Z_DIR  : {MULTI_Z_DIR}")
print(f"HAL template : {HAL_TEMPLATE.name}")

## Bucket FOVs into z-depth tiers

`N_TIERS` is deliberately a small, manually-loadable number (each tier needs
loading into HAL's parameter list by hand before the round starts -- see the
intro). FOVs are bucketed by **quantile** (equal FOV *count* per tier), not
equal z-width -- robust to a skewed thickness distribution (mostly-thin
tissue with a few much-deeper outliers would put nearly every FOV in one
equal-width bucket). Each tier's actual imaged depth is the **maximum**
`z_needed_um` assigned to it, never the quantile edge itself, so every FOV
in a tier gets at least as much depth as it needs -- never less.

In [ ]:
N_TIERS = 15   # how many discrete hal_config/shutter variants to generate
TIER_LABELS = [f"tier{i:02d}" for i in range(N_TIERS)]   # ascending depth order (tier00 = shallowest)

z_per_fov_path = METADATA_DIR / "z_per_fov_table.csv"
if not z_per_fov_path.exists():
    raise FileNotFoundError(f"{z_per_fov_path} not found -- run notebook 04 first.")
z_per_fov = pd.read_csv(z_per_fov_path)

z_needed = z_per_fov["z_needed_um"].dropna()
if len(z_needed) == 0:
    raise ValueError("No FOV has a z_needed_um value -- check notebook 04's THRESHOLD/results.")

quantile_edges = np.unique(np.quantile(z_needed, np.linspace(0, 1, N_TIERS + 1)))
if len(quantile_edges) - 1 < N_TIERS:
    print(f"WARNING: only {len(quantile_edges) - 1} distinct z_needed_um quantile edge(s) "
          f"(requested {N_TIERS} tiers) -- some FOVs share an identical z_needed_um. "
          f"Reducing to {len(quantile_edges) - 1} tier(s).")
    TIER_LABELS = TIER_LABELS[: len(quantile_edges) - 1]

z_per_fov["tier"] = pd.cut(z_per_fov["z_needed_um"], bins=quantile_edges,
                            labels=TIER_LABELS, include_lowest=True)

# FOVs with no z_needed_um at all (no detected signal in notebook 04) get the
# SHALLOWEST tier: verified (via notebook 04's section 9 mosaic) to be real
# tissue-free border FOVs, not missed signal -- so there is no tissue to lose
# by trimming short, and imaging the full stack would only waste time/data.
# Re-check that mosaic before relying on this if a future experiment's
# no-signal FOVs haven't been visually confirmed the same way.
n_no_signal = int(z_per_fov["tier"].isna().sum())
z_per_fov["tier"] = z_per_fov["tier"].astype(object).where(z_per_fov["tier"].notna(), TIER_LABELS[0])

tier_depth_um = z_per_fov.groupby("tier")["z_needed_um"].max().to_dict()
for tier in TIER_LABELS:
    if tier not in tier_depth_um or pd.isna(tier_depth_um.get(tier)):
        tier_depth_um[tier] = z_per_fov.loc[z_per_fov["tier"] == tier, "z_needed_um"].max()

print(z_per_fov["tier"].value_counts().reindex(TIER_LABELS))
print()
for tier in TIER_LABELS:
    print(f"  {tier:8s}: up to {tier_depth_um[tier]:.1f} um")
if n_no_signal:
    print(f"\n{n_no_signal} FOV(s) had no detected signal in notebook 04 -- "
          f"assigned the shallowest tier ({TIER_LABELS[0]}) as confirmed tissue-free "
          f"(verified via notebook 04's section 9 mosaic).")

## Generate one hal_config + shutter per tier

Same imaging-sequence parameters as notebook 01's bits block (`z_bead`,
`z_min`, `z_step`, `bead_seq`/`color_seq`/`end_seq`, scan/return mode) --
**keep these in sync with notebook 01** -- only `z_max` varies, per tier.

In [ ]:
z_bead    = 0
z_min     = 0.5
z_step    = 0.5
bead_seq  = [488]
color_seq = [650, 560]
end_seq   = [488]

SCAN_MODE     = "interleaved"
Z_RETURN_MODE = "progressive"
RETURN_STEP   = 5

tier_hal_configs = {}   # tier label -> {hal_stem, n_frames, z_max_um}
for tier in TIER_LABELS:
    z_max = float(tier_depth_um[tier])
    z_pos = np.arange(z_min, z_max + 1, z_step)

    frame_table = get_frame_table(z_bead, bead_seq, color_seq, end_seq, z_pos,
                                   microscope=MICROSCOPE, scan_mode=SCAN_MODE,
                                   z_return_mode=Z_RETURN_MODE, return_step=RETURN_STEP)
    name = get_color_sequence_name(frame_table, scan_mode=SCAN_MODE)

    ft_path = METADATA_DIR / frame_table_filename("bits", name, tier=tier)
    frame_table.to_csv(ft_path)

    shutter_name = shutter_filename("bits", name, tier=tier)
    # Shutter events always use full power (POWER_DEFAULT) -- POWER only sets
    # the HAL config's <default_power> below.
    create_shutter_file(frame_table, MULTI_Z_DIR / shutter_name,
                         default_power=POWER_DEFAULT)

    hal_output = MULTI_Z_DIR / hal_config_filename(MICROSCOPE, "bits", name, tier=tier)
    create_hal_config(HAL_TEMPLATE, frame_table, shutter_name, hal_output,
                      default_power=power_dict_to_channel_list(POWER, MICROSCOPE, POWER_DEFAULT),
                      file_type=FILE_TYPE, exposure_time=EXPOSURE_TIME)

    tier_hal_configs[tier] = {
        "hal_stem": hal_output.stem, "n_frames": len(frame_table), "z_max_um": z_max,
    }
    print(f"[{tier:8s}] z_max={z_max:6.1f} um, {len(frame_table):4d} frames -> {hal_output.name}")

# Saved for notebook 06 (round_info.csv's hal_config/tissue_thickness/z_lengths
# columns) -- the deepest tier (last row, sorted ascending by z_max_um) is the
# "full/representative" hal_config kept in round_info's own hal_config column.
z_tiers_df = pd.DataFrame([{"tier": t, **info} for t, info in tier_hal_configs.items()])
z_tiers_df = z_tiers_df.sort_values("z_max_um").reset_index(drop=True)
z_tiers_path = METADATA_DIR / "z_tiers.csv"
z_tiers_df.to_csv(z_tiers_path, index=False)
print(f"\nSaved: {z_tiers_path}")
print(z_tiers_df.to_string(index=False))

## Add the per-FOV tier column to the bits positions file(s)

Dave's patched `v2Generator.py` (`misc/dave_multi_z/`) reads an optional 3rd
column per positions-file line naming which hal_config to use for that FOV.
Rewrites the existing 2-column positions file(s) **in place** (idempotent --
a line already carrying a 3rd column has it replaced, not duplicated), since
the same file is also used by the cells round, which is unaffected: its own
movie template still gets a static hal_config (see `dave.py`'s
`_add_movie`), so an extra 3rd column there is simply inert.

FOVs are matched from `positions_*.txt`'s `(x, y)` values to notebook 04's
`fov_id`s by nearest stage position (not line order), since a multi-boundary
layout's positions files don't necessarily enumerate FOVs in the same order
`ExperimentMetadata` assigns global `fov_id`s.

In [ ]:
from MERci.common.metadata import ExperimentMetadata
from MERci.common.config   import ExperimentConfig

# A minimal config, just to load the same positions/round_info metadata
# notebook 04 used, so fov_id <-> (x, y) resolves identically.
_config = ExperimentConfig(
    data_dir       = SAMPLE_DIR / "data",
    metadata_dir   = METADATA_DIR,
    analysis_dir   = SAMPLE_DIR / "analysis",
    round_info_csv = METADATA_DIR / "round_info.csv",
    positions_txt  = POSITIONS_DIR / f"positions_{POSITIONS_TAG}.txt",
)
_meta = ExperimentMetadata.load(_config.round_info_csv, _config.positions_txt, _config.data_dir)

fov_ids   = np.array(list(_meta.fovs.keys()))
positions = np.array([_meta.fovs[f].position for f in fov_ids])   # (N, 2)

fov_id_to_tier_stem = dict(zip(
    z_per_fov["fov_id"], z_per_fov["tier"].map(lambda t: tier_hal_configs[t]["hal_stem"]),
))


def nearest_fov_id(x, y, max_distance_um=1.0):
    """Match a positions-file (x, y) to notebook 04's fov_id by nearest stage
    position -- robust to positions files not enumerating FOVs in fov_id order
    (true for a multi-boundary layout's per-segment files). Raises if the
    closest match is implausibly far, since that means this positions file
    doesn't actually belong to this experiment's metadata."""
    d = np.hypot(positions[:, 0] - x, positions[:, 1] - y)
    i = int(np.argmin(d))
    if d[i] > max_distance_um:
        raise ValueError(
            f"No FOV within {max_distance_um} um of ({x}, {y}) -- closest is "
            f"{d[i]:.2f} um away (fov_id {fov_ids[i]}). Positions file/metadata mismatch?"
        )
    return int(fov_ids[i])


TRANSIT_TOKEN = "transit"   # positions files with this in the name are skipped -- blank frames, no tiering
positions_files = sorted(
    p for p in POSITIONS_DIR.glob(f"positions_{POSITIONS_TAG}*.txt")
    if TRANSIT_TOKEN not in p.name
)
print(f"Positions file(s) to tag with tier column: {[p.name for p in positions_files]}")

for pfile in positions_files:
    lines = pfile.read_text().splitlines()
    new_lines = []
    n_tagged = 0
    for line in lines:
        stripped = line.strip()
        if not stripped or stripped.startswith("#"):
            new_lines.append(line)
            continue
        fields = stripped.split(",")
        x, y = float(fields[0]), float(fields[1])
        fov_id = nearest_fov_id(x, y)
        tier_stem = fov_id_to_tier_stem.get(fov_id)
        if tier_stem is None:
            raise ValueError(f"fov_id {fov_id} ({pfile.name}) has no tier assignment -- "
                              f"check z_per_fov_table.csv/notebook 04.")
        new_lines.append(f"{x},{y},{tier_stem}")
        n_tagged += 1
    pfile.write_text("\n".join(new_lines) + "\n")
    print(f"  {pfile.name}: {n_tagged} FOV(s) tagged with a tier hal_config")